In [4]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv


In [1]:
!pip install lightgbm catboost xgboost pygeohash category_encoders -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.9 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np

import pygeohash as pgh

from category_encoders import TargetEncoder

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [6]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape)
print(test.shape)

train.head()

(77299, 11)
(41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [7]:
train.isnull().sum()

,0
Index,0
geohash,0
day,0
timestamp,0
demand,0
RoadType,600
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,2495


In [8]:
train["RoadType"] = train["RoadType"].fillna("Unknown")
test["RoadType"] = test["RoadType"].fillna("Unknown")

train["Weather"] = train["Weather"].fillna("Unknown")
test["Weather"] = test["Weather"].fillna("Unknown")

train["Temperature"] = train["Temperature"].fillna(
    train["Temperature"].median()
)

test["Temperature"] = test["Temperature"].fillna(
    test["Temperature"].median()
)

In [9]:
def extract_time_features(df):

    df["hour"] = (
        df["timestamp"]
        .str.split(":")
        .str[0]
        .astype(int)
    )

    df["minute"] = (
        df["timestamp"]
        .str.split(":")
        .str[1]
        .astype(int)
    )

    df["time_slot"] = (
        df["hour"]*4 +
        df["minute"]//15
    )

    return df

train = extract_time_features(train)
test = extract_time_features(test)

In [10]:
rush_hours = [7,8,9,17,18,19]

train["rush_hour"] = train["hour"].isin(rush_hours).astype(int)
test["rush_hour"] = test["hour"].isin(rush_hours).astype(int)

In [11]:
def decode_geohash(g):

    try:
        lat, lon = pgh.decode(g)
        return pd.Series([lat, lon])

    except:
        return pd.Series([np.nan, np.nan])

In [12]:
train[["lat","lon"]] = train["geohash"].apply(decode_geohash)

test[["lat","lon"]] = test["geohash"].apply(decode_geohash)

In [13]:
for df in [train,test]:

    df["lat_lon_product"] = (
        df["lat"] * df["lon"]
    )

    df["lat_squared"] = (
        df["lat"]**2
    )

    df["lon_squared"] = (
        df["lon"]**2
    )

In [14]:
cat_cols = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]

In [15]:
# Target encoding will be performed INSIDE each fold to avoid leakage
cat_cols = [c for c in cat_cols if c in train.columns]


In [16]:
drop_cols = [
    "Index",
    "timestamp",
    "demand"
]

features = [
    col for col in train.columns
    if col not in drop_cols
]

X = train[features]
y = train["demand"]

X_test = test[features]

print(len(features))

17


In [19]:

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Initialize models before the loop
lgb = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    random_state=42
)

cat = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    verbose=0,
    random_state=42
)

xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    random_state=42
)

oof_lgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

test_lgb = np.zeros(len(test))
test_cat = np.zeros(len(test))
test_xgb = np.zeros(len(test))

# Redefine features to exclude 'demand', 'Index', and 'timestamp'
modeling_features = [col for col in train.columns if col not in ['demand', 'Index', 'timestamp']]

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f'Fold {fold+1}')

    X_tr = train.iloc[tr_idx][modeling_features].copy()
    X_val = train.iloc[val_idx][modeling_features].copy()

    y_tr = train.iloc[tr_idx]['demand']
    y_val = train.iloc[val_idx]['demand']

    X_test_fold = test[modeling_features].copy()

    encoder = TargetEncoder(cols=cat_cols)

    X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols], y_tr)
    X_val[cat_cols] = encoder.transform(X_val[cat_cols])
    X_test_fold[cat_cols] = encoder.transform(X_test_fold[cat_cols])

    lgb.fit(X_tr, y_tr)
    cat.fit(X_tr, y_tr)
    xgb.fit(X_tr, y_tr)

    oof_lgb[val_idx] = lgb.predict(X_val)
    oof_cat[val_idx] = cat.predict(X_val)
    oof_xgb[val_idx] = xgb.predict(X_val)

    test_lgb += lgb.predict(X_test_fold) / kf.n_splits
    test_cat += cat.predict(X_test_fold) / kf.n_splits
    test_xgb += xgb.predict(X_test_fold) / kf.n_splits

ensemble_oof = 0.4*oof_lgb + 0.3*oof_cat + 0.3*oof_xgb
ensemble_test = 0.4*test_lgb + 0.3*test_cat + 0.3*test_xgb


Fold 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004875 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1084
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 17
[LightGBM] [Info] Start training from score 0.093784
Fold 2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1084
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 17
[LightGBM] [Info] Start training from score 0.093797
Fold 3
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006372 seconds.
You can set `force_row_wise=true` to remove the overhead.
And i

In [21]:
lgb_oof = np.zeros(len(X))
lgb_pred = np.zeros(len(X_test))

# Define modeling features similar to the ensemble cell
modeling_features = [col for col in train.columns if col not in ['demand', 'Index', 'timestamp']]

for tr_idx,val_idx in kf.split(X):

    X_tr = train.iloc[tr_idx][modeling_features].copy()
    y_tr = y.iloc[tr_idx]

    X_val = train.iloc[val_idx][modeling_features].copy()

    # Prepare X_test_fold for predictions
    X_test_fold = test[modeling_features].copy()

    # Apply Target Encoding
    encoder = TargetEncoder(cols=cat_cols)
    X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols], y_tr)
    X_val[cat_cols] = encoder.transform(X_val[cat_cols])
    X_test_fold[cat_cols] = encoder.transform(X_test_fold[cat_cols])

    model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=8,
        random_state=42
    )

    model.fit(X_tr,y_tr)

    lgb_oof[val_idx] = model.predict(X_val)

    lgb_pred += model.predict(X_test_fold)/5

print("Done")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004018 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1084
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 17
[LightGBM] [Info] Start training from score 0.093784
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004132 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1084
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 17
[LightGBM] [Info] Start training from score 0.093797
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004704 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enoug

In [23]:
cat_oof = np.zeros(len(X))
cat_pred = np.zeros(len(X_test))

# Define modeling features similar to the ensemble cell
modeling_features = [col for col in train.columns if col not in ['demand', 'Index', 'timestamp']]

for tr_idx,val_idx in kf.split(X):

    X_tr = train.iloc[tr_idx][modeling_features].copy()
    y_tr = y.iloc[tr_idx]

    X_val = train.iloc[val_idx][modeling_features].copy()

    # Prepare X_test_fold for predictions
    X_test_fold = test[modeling_features].copy()

    # Apply Target Encoding
    encoder = TargetEncoder(cols=cat_cols)
    X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols], y_tr)
    X_val[cat_cols] = encoder.transform(X_val[cat_cols])
    X_test_fold[cat_cols] = encoder.transform(X_test_fold[cat_cols])

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=8,
        verbose=0,
        random_state=42 # Added random_state for consistency with other models
    )

    model.fit(X_tr,y_tr)

    cat_oof[val_idx] = model.predict(X_val)

    cat_pred += model.predict(X_test_fold)/5

print("Done")

Done


In [25]:
xgb_oof = np.zeros(len(X))
xgb_pred = np.zeros(len(X_test))

# Define modeling features similar to the ensemble cell
modeling_features = [col for col in train.columns if col not in ['demand', 'Index', 'timestamp']]

for tr_idx,val_idx in kf.split(X):

    X_tr = train.iloc[tr_idx][modeling_features].copy()
    y_tr = y.iloc[tr_idx]

    X_val = train.iloc[val_idx][modeling_features].copy()

    # Prepare X_test_fold for predictions
    X_test_fold = test[modeling_features].copy()

    # Apply Target Encoding
    encoder = TargetEncoder(cols=cat_cols)
    X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols], y_tr)
    X_val[cat_cols] = encoder.transform(X_val[cat_cols])
    X_test_fold[cat_cols] = encoder.transform(X_test_fold[cat_cols])

    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=8,
        random_state=42
    )

    model.fit(X_tr,y_tr)

    xgb_oof[val_idx] = model.predict(X_val)

    xgb_pred += model.predict(X_test_fold)/5

print("Done")

Done


In [26]:
ensemble_oof = (
    0.4*lgb_oof +
    0.3*cat_oof +
    0.3*xgb_oof
)

score = mean_absolute_error(
    y,
    ensemble_oof
)

print("MAE:",score)

MAE: 0.021686814258327487


In [27]:
final_pred = (
    0.4*lgb_pred +
    0.3*cat_pred +
    0.3*xgb_pred
)

In [28]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_pred
})

submission.head()

,Index,demand
0,0,0.056689
1,1,0.034190
2,2,0.047920
3,3,0.035437
4,4,0.047961


In [29]:
submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,Index,demand
0,0,0.056689
1,1,0.034190
2,2,0.047920
3,3,0.035437
4,4,0.047961


In [30]:
from google.colab import files

files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
from sklearn.metrics import r2_score

print("LGBM :", r2_score(y, lgb_oof))
print("CAT  :", r2_score(y, cat_oof))
print("XGB  :", r2_score(y, xgb_oof))
print("ENS  :", r2_score(y, ensemble_oof))

LGBM : 0.9478203343723609
CAT  : 0.9449548992275603
XGB  : 0.955633193975801
ENS  : 0.9515071353169411


In [32]:
from sklearn.metrics import r2_score

r2 = r2_score(y, ensemble_oof)

lb_score = max(0, 100*r2)

print("R2 Score:", r2)
print("Leaderboard Score:", lb_score)

R2 Score: 0.9515071353169411
Leaderboard Score: 95.15071353169411
